# CS 3892 / 5892 — Session 12 · CTL and Büchi automata

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ttj/cs3892-examples/blob/main/notebooks/cs3892-2026-10-06-ctl-and-buchi-automata.ipynb)

**Tuesday, October 6, 2026.** LTL judges every run; **CTL** makes the path quantifier explicit —
`E` (on *some* path) and `A` (on *every* path) — and so can say what is still **possible**, not only
what is guaranteed. Then the automata view of LTL model checking: negate the property, build a
**Büchi automaton** for the bad runs, and search the product for an **accepting cycle** — a lasso.

Everything runs **NuSMV** (free, LGPL), fetched by the setup cells, on the files in
`sessions/cs3892-2026-10-06-ctl-and-buchi-automata/`. Every verdict is asserted. The agent model is session 11's.

## Setup

Run these two once.

In [ ]:
# --- Setup: find the repo (clone on Colab) and define helpers ---------------
import os, re, subprocess, sys, pathlib

REPO_URL = "https://github.com/ttj/cs3892-examples.git"
SESSION  = "cs3892-2026-10-06-ctl-and-buchi-automata"

def _find_repo():
    """Walk up from the CWD looking for the repo; otherwise clone it."""
    here = pathlib.Path.cwd()
    for p in [here, *here.parents]:
        if (p / "sessions" / SESSION).is_dir():
            return p
    dest = pathlib.Path("/content/cs3892-examples") if pathlib.Path("/content").is_dir() \
           else pathlib.Path.cwd() / "cs3892-examples"
    if not (dest / "sessions" / SESSION).is_dir():
        print(f"$ git clone {REPO_URL} {dest}")
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(dest)], check=True)
    return dest

ROOT = _find_repo()
os.chdir(ROOT)
print("repo:", ROOT)

SMV = ROOT / "sessions" / SESSION / "smv"
PYD = ROOT / "sessions" / SESSION / "python"

def show(path):
    """Print a source file, so you can read what you are about to run."""
    path = pathlib.Path(path)
    print(f"--- {path.name} " + "-" * max(0, 60 - len(path.name)))
    print(path.read_text().rstrip())
    print()

def run(path):
    """Run one Python example and stream its output. Raises if it fails."""
    path = pathlib.Path(path)
    show(path)
    r = subprocess.run([sys.executable, str(path)], capture_output=True, text=True)
    print(r.stdout.rstrip())
    if r.returncode != 0:
        print(r.stderr.rstrip(), file=sys.stderr)
        raise RuntimeError(f"{path} failed")
    return r.stdout

print("ready — helpers: show(path), run(path)")

In [ ]:
# --- NuSMV: fetch the official FBK build if it is not already installed ------
# NuSMV is free (LGPL) from nusmv.fbk.eu. (Same cell as sessions 9 and 10.)
# 2.7.1 is tried first; it needs glibc >= 2.38 and libedit, which an older
# Colab image may lack, so the static 2.6.0 build is the fallback. Same verdicts.
import shutil

BUILDS = [("2.7.1", "https://nusmv.fbk.eu/distrib/2.7.1/NuSMV-2.7.1-linux64.tar.xz", "J"),
          ("2.6.0", "https://nusmv.fbk.eu/distrib/NuSMV-2.6.0-linux64.tar.gz",       "z")]

def _works(exe):
    """True only if this binary actually model-checks a file -- a missing
    library makes it exit 127 with an error that still contains "NuSMV"."""
    try:
        r = subprocess.run([exe, str(SMV / "01_ctl_on_the_agent.smv")],
                           capture_output=True, text=True, timeout=60)
        return "-- specification" in r.stdout
    except Exception:
        return False

NUSMV = shutil.which("NuSMV")
if not (NUSMV and _works(NUSMV)):
    NUSMV = None
    if os.geteuid() == 0 and shutil.which("apt-get"):      # Colab runs as root
        subprocess.run("apt-get install -y -qq libedit2 >/dev/null 2>&1", shell=True)
    for ver, url, z in BUILDS:
        dest = pathlib.Path.home() / ".local" / f"nusmv-{ver}"
        dest.mkdir(parents=True, exist_ok=True)
        subprocess.run(f"curl -sSL {url} | tar -x{z} -C {dest} --strip-components=1",
                       shell=True, check=True)
        if _works(str(dest / "bin" / "NuSMV")):
            NUSMV = str(dest / "bin" / "NuSMV")
            break
        print(f"NuSMV {ver} will not start on this machine -- trying the next build")
assert NUSMV, "could not install NuSMV"
print("NuSMV:", NUSMV)

def smv(path):
    """Run NuSMV on a file in batch mode; return its output minus the banner."""
    r = subprocess.run([NUSMV, str(path)], capture_output=True, text=True)
    return "\n".join(l for l in (r.stdout + r.stderr).splitlines() if not l.startswith("***"))

def results(out):
    """Split NuSMV output into {printed formula: (verdict, counterexample text)}.
    NuSMV does not print verdicts in file order, so match them by formula."""
    res, key = {}, None
    for line in out.splitlines():
        m = re.match(r"-- (?:specification|invariant)\s+(.*?)\s+is (true|false)$", line)
        if m:
            key = re.sub(r"\s+", " ", m.group(1)).strip()
            res[key] = [m.group(2), ""]
        elif key and res[key][0] == "false":
            res[key][1] += line + "\n"
    return {k: tuple(v) for k, v in res.items()}

def verdicts(path):
    """Print one line per property, and return {formula: verdict}."""
    out = smv(path)
    res = results(out)
    for k, (v, _) in res.items():
        print(f"{v:6} {k}")
    return {k: v for k, (v, _) in res.items()}

print("ready -- smv(path), results(out), verdicts(path)")

## 1. CTL on the agent: E versus A

Slides 18–22. From `plan` the agent may ask or abandon; from `ask` the human says yes or no. The
runs branch, so every CTL formula below is a question about the **tree** of runs from `plan`.

| formula | reads | verdict |
|---|---|---|
| `EX step = ask` / `AX step = ask` | some / every next state asks | true / **false** |
| `EF del` / `AF del` | some / every path deletes eventually | true / **false** |
| `EG !del` / `AG !del` | some / every path never deletes | true / **false** |
| `AG EF del` | from everywhere, a delete is still **possible** | true |
| `AG AF del` | from everywhere, a delete is **promised** | **false** |

In [ ]:
F = SMV / "01_ctl_on_the_agent.smv"
show(F)
out = smv(F)
res = results(out)
for k, (v, _) in res.items():
    print(f"{v:6} {k}")
expect = {"EX step = ask": "true", "AX step = ask": "false", "EF del": "true", "AF del": "false",
          "EG !del": "true", "AG !del": "false", "AG (EF del)": "true", "AG (AF del)": "false",
          "AG (del -> approved)": "true", "G ( F del)": "false", "G (del -> approved)": "true"}
assert {k: v for k, (v, _) in res.items()} == expect, res.keys()
print("\n--- AF del is false: the path that abandons every task (a lasso)")
print(res["AF del"][1])
assert "Loop starts here" in res["AF del"][1] and "step = act" not in res["AF del"][1]

**A false `A`-formula comes back with a path** — the run where it fails. A false `E`-formula has no
single run to show (no one run refutes "*some* path"), and NuSMV prints just the starting state.

## 2. CTL's until

Slide 21. `E [p U q]` and `A [p U q]` — in NuSMV syntax, with square brackets. (smvis cannot parse
these, which is why they live in their own file.)

In [ ]:
F = SMV / "02_ctl_until.smv"
show(F)
v = {k: x for k, (x, _) in results(smv(F)).items()}
for k, x in v.items():
    print(f"{x:6} {k}")
assert v == {"E [ !del U del ]": "true", "A [ !del U del ]": "false",
             "A [ !del U (del | step = abort) ]": "true"}, v

## 3. LTL and CTL say different things

Slides 23–24. On this three-state machine `F G p` (LTL) is **true** but its "obvious" CTL reading
`AF AG p` is **false** — and `AG EF (s = s2)` (CTL) is **true** while no LTL formula can say "still
possible". Neither logic contains the other.

In [ ]:
F = SMV / "03_ltl_vs_ctl.smv"
show(F)
v = {k: x for k, (x, _) in results(smv(F)).items()}
for k, x in v.items():
    print(f"{x:6} {k}")
assert v == {"AF (AG p)": "false", "AG (EF s = s2)": "true", "F ( G p)": "true", "F s = s2": "false"}, v

## 4. Büchi automata: model checking LTL by hand

Slides 27–30. `04_buchi_emptiness.py` does what NuSMV does inside, small enough to read: a Büchi
automaton for the **negated** property, the product with the agent, and a search for a reachable
**accepting cycle**. Found → a lasso, the counterexample. Not found → the property holds.
The cell after it checks that NuSMV agrees on every case.

In [ ]:
run(PYD / "04_buchi_emptiness.py")

In [ ]:
# NuSMV on the same three questions -- the buggy agent is session 11's file.
agent = results(smv(SMV / "01_ctl_on_the_agent.smv"))
buggy = results(smv(ROOT / "sessions" / "cs3892-2026-10-01-linear-temporal-logic" / "smv" / "03_agent_approval_buggy.smv"))
assert agent["G ( F del)"][0] == "false"            # fixed agent: G F del fails
assert agent["G (del -> approved)"][0] == "true"    # fixed agent: safety holds
assert buggy["G (del -> approved)"][0] == "false"   # buggy agent: safety fails
print("NuSMV agrees with the hand-built product on all three")

## 5. In smvis

[bit.ly/fmaiv_smvis](https://bit.ly/fmaiv_smvis): paste `01_ctl_on_the_agent.smv`, **Parse**,
**Compute All**, **Check All Specs**. Then under **LTL Buchi Analysis** pick `G (F (del))` and
**Compose & Analyze**: smvis negates it to `F (G (!del))`, builds a 2-state Büchi automaton, forms
the product and finds an accepting cycle — the same lasso shape as section 4, drawn on the graph.

## Try it yourself

- In `01`, is `AG (step = ask -> EF del)` true? `AG (step = ask -> AF del)`? Predict, then run.
- Write "the agent can always get back to planning" in CTL. Is it true?
- In `03`, add `SPEC AG AF p`. True or false — and how does it differ from `AF AG p`?